# Análise do Efeito Fotoelétrico

Este notebook realiza a análise experimental do efeito fotoelétrico, determinando a constante de Planck e a função trabalho do material.

## Metodologia

O notebook implementa 4 métodos diferentes para determinação do potencial de corte (V₀):
- **Método 1**: Cruzamento com zero
- **Método 2**: Intersecção de tangentes assintóticas
- **Método 3**: Intersecção de curvas de diferentes intensidades
- **Método 4**: Análise estatística da segunda derivada (onset)

In [ ]:
# Bibliotecas necessárias
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
import scipy.stats as stats
import itertools

# Configuração do diretório de dados
data_dir = r"C:\Users\gabri\Documents\PROJETOS\POLI\Lab Física\Fotoelétrico\attachments"
sem_file = os.path.join(data_dir, "sem.csv")

# Carregar dados de ruído de fundo (sem luz)
df_sem = pd.read_csv(sem_file, sep=";", decimal=",")

## 1. Importações e Configuração

In [ ]:
# Frequências das cores (em 10^14 Hz)
frequencias = {
    "Vermelho": 4.875,
    "Amarelo": 5.187,  # Média das linhas do dubleto (5.196 e 5.178)
    "Verde": 5.490,
    "Azul": 6.879,
    "Violeta": 7.409,
    "Ultra-Violeta": 8.213,
}

# Constante de Planck teórica (eV·s)
h = 4.13

# Dados teóricos/esperados para V₀ (V)
expected_data = {
    "Ultra-Violeta": 2.182,
    "Violeta": 1.850,
    "Azul": 1.631,
    "Verde": 1.057,
    "Amarelo": 0.932,
    "Vermelho": 0.803,
}

# Constantes de processamento
SAVGOL_WINDOW = 15
SAVGOL_POLY = 3
SCALE = 1e10
MIN_DATA_POINTS = 20
V0_RANGE = (-5, 2)  # Range de validade física para V₀
SLOPE_THRESHOLD = 1e-5
SIGMA_INST = 0.005  # Incerteza instrumental (5 mV)

## 2. Constantes e Dados Experimentais

In [ ]:
# Configuração das cores (ordem: Vermelho, Amarelo, Verde, Azul, Violeta, UV)
colors_info = [
    {
        "name": "Vermelho",
        "prefix": "ver",
        "letter": "(f)",
        "xlim": (-1.5, 1.0),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Amarelo",
        "prefix": "amar",
        "letter": "(a)",
        "xlim": (-1.5, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Verde",
        "prefix": "verde",
        "letter": "(b)",
        "xlim": (-2.5, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Azul",
        "prefix": "azul",
        "letter": "(c)",
        "xlim": (-3.0, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Violeta",
        "prefix": "vio",
        "letter": "(d)",
        "xlim": (-3.5, 0.5),
        "ylim": (-1.5, 1.5),
    },
    {
        "name": "Ultra-Violeta",
        "prefix": "UV",
        "letter": "(e)",
        "xlim": (-4.0, 0.5),
        "ylim": (-1.5, 1.5),
    },
]

# Cores das intensidades
intensity_colors = {
    100: "tab:green",
    80: "#f9d71c",
    60: "tab:blue",
    40: "tab:red",
    20: "tab:gray",
}

intensities = [100, 80, 60, 40, 20]

# Estilo global dos gráficos
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.axisbelow"] = True

## 3. Configuração de Cores e Estilo Gráfico

In [ ]:
# Gráficos das medições

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, info in enumerate(colors_info):
    ax = axes_flat[i]

    # Ordenar intensidades para legenda (descendente conforme imagem)
    intensities_to_plot = [100, 80, 60, 40, 20]

    for intensity in intensities_to_plot:
        file_path = os.path.join(data_dir, f"{info['prefix']}-{intensity}.csv")

        df = pd.read_csv(file_path, sep=";", decimal=",")

        corrente = df["Corrente [A]"]
        corrente = corrente - df_sem["Corrente [A]"]

        corrente_plot = corrente * 1e10
        tensao = df["Tensao [V]"]

        ax.plot(
            tensao,
            corrente_plot,
            label=f"{intensity}%",
            color=intensity_colors.get(intensity, "black"),
            linewidth=1.5,
        )

    # Configurações estéticas por subplot
    ax.set_xlabel("Potencial Aplicado (V)", fontsize=12)
    ax.set_ylabel(r"Corrente ($x10^{-10}$ A)", fontsize=12)

    # Letra (a, b, c...) no topo direito
    ax.text(
        0.95,
        0.95,
        info["letter"],
        transform=ax.transAxes,
        fontsize=16,
        va="top",
        ha="right",
    )

    # Nome da cor no canto inferior direito
    ax.text(
        0.98,
        0.02,
        info["name"],
        transform=ax.transAxes,
        fontsize=14,
        va="bottom",
        ha="right",
    )

    # Grid e ticks
    ax.grid(True, which="major", linestyle="--", color="gray", alpha=0.6)
    ax.minorticks_on()
    ax.grid(True, which="minor", linestyle=":", color="gray", alpha=0.3)

    # Legenda
    ax.legend(
        loc="upper left", fontsize=10, frameon=True, edgecolor="black", fancybox=False
    )

    # Limites de escala
    if info["xlim"]:
        ax.set_xlim(info["xlim"])
    if info["ylim"]:
        ax.set_ylim(info["ylim"])

    ax.tick_params(axis="both", which="major", labelsize=10)

plt.tight_layout()
plt.show()

## 4. Visualização dos Dados Brutos

In [ ]:
def validate_input_data(v, i_vals):
    """Valida os dados de entrada para todos os métodos."""
    if len(v) != len(i_vals):
        raise ValueError("Voltage and current arrays must have same length")
    if len(v) < MIN_DATA_POINTS:
        return False
    if not np.all(np.isfinite(v)) or not np.all(np.isfinite(i_vals)):
        return False
    return True


def method_1_zero_crossing(v, i_vals):
    """
    Método 1: Identificar o V₀ pelo cruzamento com zero.
    
    Identifica o ponto onde a corrente cruza o eixo zero (I=0),
    escolhendo o cruzamento mais próximo de zero.
    """
    i_s = i_vals * SCALE
    signs = np.sign(i_s)
    sign_changes = ((np.roll(signs, 1) - signs) != 0).astype(int)
    sign_changes[0] = 0
    indices = np.where(sign_changes == 1)[0]

    best_v0 = np.nan
    min_distance_to_zero = float("inf")

    for idx in indices:
        if idx == 0 or idx >= len(v):
            continue
        y0, y1 = i_s[idx - 1], i_s[idx]
        x0, x1 = v[idx - 1], v[idx]
        
        if abs(y1 - y0) < 1e-15:
            continue
            
        v_cross = x0 - y0 * (x1 - x0) / (y1 - y0)

        distance = abs(v_cross)
        if distance < min_distance_to_zero and V0_RANGE[0] < v_cross < V0_RANGE[1]:
            min_distance_to_zero = distance
            best_v0 = v_cross

    return best_v0


def method_2_tangents(v, i_vals, window_size=10, bg_offset=10, bg_width=40):
    """
    Método 2: Intersecção das retas assintóticas.
    
    Calcula a intersecção entre a reta tangente na região de subida
    do sinal e a reta de fundo (background).
    """
    i_s = i_vals * SCALE
    i_smooth = savgol_filter(i_s, 15, 3)
    di = np.gradient(i_smooth)
    max_deriv_idx = np.argmax(di)

    # Região de subida (rise)
    s_r = max(0, max_deriv_idx - window_size)
    e_r = min(len(v), max_deriv_idx + window_size)
    if s_r >= e_r:
        return np.nan

    res_rise = stats.linregress(v[s_r:e_r], i_smooth[s_r:e_r])

    # Região de background
    e_b = max(5, s_r - bg_offset)
    s_b = max(0, e_b - bg_width)

    if e_b - s_b < 5:
        return np.nan

    res_bg = stats.linregress(v[s_b:e_b], i_smooth[s_b:e_b])

    if abs(res_rise.slope - res_bg.slope) < SLOPE_THRESHOLD:
        return np.nan

    intersection = (res_bg.intercept - res_rise.intercept) / (
        res_rise.slope - res_bg.slope
    )

    if not (V0_RANGE[0] < intersection < V0_RANGE[1]):
        return np.nan

    return intersection


def method_3_intersection_curves(curves_list, min_curves=2, window_half=8):
    """
    Método 3: Intersecção das curvas de diferentes intensidades.
    
    Determina V₀ pela intersecção das retas tangentes às curvas
    de diferentes intensidades luminosas.
    """
    if len(curves_list) < min_curves:
        return np.nan

    lines = []
    for v, i_vals in curves_list:
        if len(v) < 20:
            continue

        i_s = i_vals * SCALE
        i_smooth = savgol_filter(i_s, min(15, len(i_s) // 3), 3)
        di = np.gradient(i_smooth)
        max_idx = np.argmax(di)

        w = min(window_half, max_idx // 2, (len(v) - max_idx) // 2)
        s, e = max(0, max_idx - w), min(len(v), max_idx + w)

        if e - s >= 5:
            res = stats.linregress(v[s:e], i_smooth[s:e])
            if abs(res.slope) > SLOPE_THRESHOLD:
                lines.append((res.slope, res.intercept))

    if len(lines) < 2:
        return np.nan

    v0_candidates = []
    for (a1, b1), (a2, b2) in itertools.combinations(lines, 2):
        if abs(a1 - a2) > 1e-2:
            v_int = (b2 - b1) / (a1 - a2)
            if V0_RANGE[0] < v_int < V0_RANGE[1]:
                v0_candidates.append(v_int)

    return np.median(v0_candidates) if len(v0_candidates) >= 2 else np.nan


def method_4_statistical_onset(
    v, i_vals, noise_region=(-9, -3), confidence_sigma=3, min_noise_points=10
):
    """
    Método 4: V₀ estatístico pela análise da segunda derivada.
    
    Determina o ponto onde o sinal da segunda derivada sai 
    estatisticamente da região de ruído.
    """
    if len(v) < 50:
        return np.nan

    window_size = min(21, len(i_vals) // 5)
    if window_size < 5:
        return np.nan

    i_s = i_vals * SCALE
    i_smooth = savgol_filter(i_s, window_size, 3)

    di = np.gradient(i_smooth, v)
    d2i = np.gradient(di, v)

    mask_noise = (v >= noise_region[0]) & (v <= noise_region[1])

    if np.sum(mask_noise) < min_noise_points:
        return np.nan

    v_noise = v[mask_noise]
    d2i_noise = d2i[mask_noise]

    try:
        res = stats.linregress(v_noise, d2i_noise)
        predicted_noise = res.intercept + res.slope * v_noise
        residuals = d2i_noise - predicted_noise

        # Remover outliers usando método IQR
        Q1, Q3 = np.percentile(residuals, [25, 75])
        IQR = Q3 - Q1
        outlier_mask = (residuals >= Q1 - 1.5 * IQR) & (residuals <= Q3 + 1.5 * IQR)

        if np.sum(outlier_mask) < min_noise_points // 2:
            return np.nan

        std_noise = np.std(residuals[outlier_mask])

    except (ValueError, np.linalg.LinAlgError):
        return np.nan

    search_indices = np.where(v > noise_region[1])[0]

    for idx in search_indices:
        val_pred = res.intercept + res.slope * v[idx]
        val_real = d2i[idx]

        if np.abs(val_real - val_pred) > confidence_sigma * std_noise:
            return v[idx]

    return np.nan

## 5. Métodos de Determinação de V₀

### 5.1 Funções dos Métodos

In [ ]:
# --- Execução da Análise ---

resultados_list = []
print(
    f"{'Cor':<15} | {'M1: Zero':<10} | {'M2: Tan':<10} | {'M3: Inter':<10} | {'M4: 2Der':<10}"
)
print("-" * 65)

for info in colors_info:
    cor_name = info["name"]
    prefix = info["prefix"]

    curves_data = []
    v_main, i_main = None, None

    # Coleta todas as intensidades disponíveis
    for intensity in [100, 80, 60, 40, 20]:
        fpath = os.path.join(data_dir, f"{prefix}-{intensity}.csv")
        if os.path.exists(fpath):
            df_temp = pd.read_csv(fpath, sep=";", decimal=",")
            corrente = df_temp["Corrente [A]"].values
            if df_sem is not None and len(corrente) == len(df_sem):
                corrente = corrente - df_sem["Corrente [A]"].values
            tensao = df_temp["Tensao [V]"].values

            curves_data.append((tensao, corrente))
            if intensity == 100 or v_main is None:
                v_main, i_main = tensao, corrente

    if v_main is not None:
        v0_m1 = method_1_zero_crossing(v_main, i_main)
        v0_m2 = method_2_tangents(v_main, i_main)
        v0_m3 = method_3_intersection_curves(curves_data)
        v0_m4 = method_4_statistical_onset(v_main, i_main)

        # Escolha do melhor V0 para a regressão de Planck
        v0_final = v0_m4

        resultados_list.append(
            {
                "Cor": cor_name,
                "V0_M1": v0_m1,
                "V0_M2": v0_m2,
                "V0_M3": v0_m3,
                "V0_M4": v0_m4,
                "V0_medio": v0_final,
                "Incerteza": 0.05,
            }
        )
        print(
            f"{cor_name:<15} | {v0_m1:8.4f} | {v0_m2:8.4f} | {v0_m3:8.4f} | {v0_m4:8.4f}"
        )

# Create results DataFrame
df_resultados = pd.DataFrame(resultados_list)

# --- Comparação com Valores Esperados e Gráfico de Desempenho dos Métodos ---

# Dados teóricos/esperados fornecidos
expected_data = {
    "Ultra-Violeta": 2.182,
    "Violeta": 1.850,
    "Azul": 1.631,
    "Verde": 1.057,
    "Amarelo": 0.932,
    "Vermelho": 0.803,
}

# Preparar DataFrame de comparação (aplicando módulo)
df_comparison = df_resultados.copy()
metodos = ["V0_M1", "V0_M2", "V0_M3", "V0_M4"]

# Apply absolute value to all method results
for metodo in metodos:
    df_comparison[metodo] = df_comparison[metodo].abs()

# Add expected values and frequencies
df_comparison["Esperado"] = df_comparison["Cor"].map(expected_data)
df_comparison["f"] = df_comparison["Cor"].map(frequencias)

# Remove rows without expected values and sort by frequency
df_comparison = df_comparison.dropna(subset=["Esperado"]).sort_values("f")

# Plotagem da Comparação
plt.figure(figsize=(12, 7))

# Plot each method
plt.plot(
    df_comparison["f"],
    df_comparison["Esperado"],
    "k--",
    marker="s",
    label="Teórico (Esperado)",
    linewidth=2,
    markersize=8,
)
plt.plot(
    df_comparison["f"],
    df_comparison["V0_M1"],
    "o-",
    label="M1: Cruzamento Zero",
    alpha=0.7,
)
plt.plot(
    df_comparison["f"],
    df_comparison["V0_M2"],
    "o-",
    label="M2: Intersecção Tangentes",
    alpha=0.8,
    linewidth=2,
)
plt.plot(
    df_comparison["f"],
    df_comparison["V0_M3"],
    "o-",
    label="M3: Intersecção Curvas",
    alpha=0.7,
)
plt.plot(
    df_comparison["f"], df_comparison["V0_M4"], "o-", label="M4: 2ª Derivada", alpha=0.7
)

# Estética do gráfico
plt.title("Comparação dos Métodos de Determinação de $V_0$", fontsize=16)
plt.xlabel("Frequência ($10^{14}$ Hz)", fontsize=14)
plt.ylabel("$|V_0|$ (V)", fontsize=14)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=10)

# Incluir anotação das cores no eixo X ou pontos
for i, row in df_comparison.iterrows():
    plt.annotate(
        row["Cor"],
        (row["f"], row["Esperado"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

# Calculate relative errors (%)
for metodo in metodos:
    df_comparison[f"Erro_{metodo}(%)"] = (
        np.abs(
            (df_comparison[metodo] - df_comparison["Esperado"])
            / df_comparison["Esperado"]
        )
        * 100
    )

# Display error table
print("\n--- Erro Relativo (%) em relação ao Esperado ---")
columns_show = ["Cor"] + [f"Erro_{metodo}(%)" for metodo in metodos]
print(df_comparison[columns_show].to_string(index=False, float_format="{:.2f}".format))

# Calculate and display average error for each method
print("\n--- Erro Médio (%) por Método ---")
for metodo in metodos:
    erro_medio = df_comparison[f"Erro_{metodo}(%)"].mean()
    print(f"{metodo}: {erro_medio:.2f}%")

### 5.2 Aplicação dos Métodos e Comparação

In [ ]:
def plot_method_4_stacked_origin_style(
    curves_list, cor_name, noise_region=(-9, -3), sigma_multiplier=1.5
):
    """
    Gera um gráfico estilo 'Origin' com subplots empilhados para o Método 4.
    Calcula o V₀ individual para cada curva e desenha a linha média global.

    Args:
        curves_list: Lista de tuplas (v, i, label)
        cor_name: Nome da cor para o título
        noise_region: Tupla (v_min, v_max) para ajuste do ruído
        sigma_multiplier: Multiplicador do desvio padrão para banda de predição
    """
    n_curves = len(curves_list)
    fig, axes = plt.subplots(
        nrows=n_curves, ncols=1, figsize=(10, 2 * n_curves), sharex=True
    )

    if n_curves == 1:
        axes = [axes]

    plt.subplots_adjust(hspace=0)

    v0_detected_list = []
    processed_curves = []

    # Primeira passagem: Cálculos
    for v, i_vals, label in curves_list:
        i_s = i_vals * SCALE
        window_size = max(15, min(21, len(i_vals) // 5))
        if window_size % 2 == 0:
            window_size += 1

        i_smooth = savgol_filter(i_s, window_size, 3)
        di = np.gradient(i_smooth, v)
        d2i = np.gradient(di, v)

        mask_noise = (v >= noise_region[0]) & (v <= noise_region[1])
        v_noise = v[mask_noise]
        d2i_noise = d2i[mask_noise]

        if len(v_noise) < 10:
            processed_curves.append(None)
            continue

        res = stats.linregress(v_noise, d2i_noise)

        predicted_noise_region = res.intercept + res.slope * v_noise
        residuals = d2i_noise - predicted_noise_region
        std_noise = np.std(residuals)

        search_region = v > noise_region[1]
        v_search = v[search_region]
        d2i_search = d2i[search_region]

        v0_found = np.nan
        predicted_full = res.intercept + res.slope * v_search
        diff = np.abs(d2i_search - predicted_full)
        threshold = sigma_multiplier * std_noise

        out_indices = np.where(diff > threshold)[0]

        if len(out_indices) > 0:
            idx_v0 = out_indices[0]
            v0_found = v_search[idx_v0]
            v0_detected_list.append(v0_found)

        processed_curves.append(
            {
                "v": v,
                "d2i": d2i,
                "res": res,
                "std_noise": std_noise,
                "v0": v0_found,
                "label": label,
                "threshold_val": threshold,
            }
        )

    v0_global_mean = np.mean(v0_detected_list) if v0_detected_list else np.nan

    # Segunda passagem: Plotagem
    for idx, (ax, data) in enumerate(zip(axes, processed_curves)):
        if data is None:
            ax.text(
                0.5, 0.5, "Dados Insuficientes", transform=ax.transAxes, ha="center"
            )
            continue

        v = data["v"]
        d2i = data["d2i"]
        res = data["res"]
        threshold = data["threshold_val"]

        # Plot do sinal
        ax.plot(v, d2i, color="#000080", linewidth=1.5, zorder=2, label="2ª Derivada")

        # Fit linear
        x_fit = np.array([v.min(), v.max()])
        y_fit = res.intercept + res.slope * x_fit
        ax.plot(x_fit, y_fit, color="black", linewidth=1, zorder=3)

        # Banda de confiança
        y_upper = y_fit + threshold
        y_lower = y_fit - threshold
        ax.fill_between(x_fit, y_lower, y_upper, color="gray", alpha=0.3, zorder=1)

        # Linha média global
        if not np.isnan(v0_global_mean):
            ax.axvline(v0_global_mean, color="red", linewidth=2, zorder=5)

        # Marcadores da região de ruído
        ax.axvline(
            noise_region[0], color="black", linestyle="-", linewidth=0.8, alpha=0.5
        )
        ax.axvline(
            noise_region[1], color="black", linestyle="-", linewidth=0.8, alpha=0.5
        )

        # Label da intensidade
        ax.text(
            0.02,
            0.85,
            f"Intensidade: {data['label']}",
            transform=ax.transAxes,
            fontsize=10,
            fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
        )

        ax.grid(True, linestyle=":", alpha=0.5)
        ax.set_xlim(-10, 1.5)

        # Ajuste automático do eixo Y
        mask_view = (v > -5) & (v < 1)
        if np.any(mask_view):
            y_roi = d2i[mask_view]
            ymin, ymax = y_roi.min(), y_roi.max()
            margin = (ymax - ymin) * 0.5
            ax.set_ylim(ymin - margin, ymax + margin)

        if idx < n_curves - 1:
            ax.tick_params(labelbottom=False)

    # Título e labels
    axes[0].set_title(
        f"Determinação de V₀ (Método 4 - Empilhado) para {cor_name}\nMédia V₀ = {v0_global_mean:.3f} V",
        fontsize=14,
        pad=20,
    )

    fig.text(
        0.04,
        0.5,
        r"Segunda Derivada ($d^2I/dV^2$)",
        va="center",
        rotation="vertical",
        fontsize=14,
    )
    axes[-1].set_xlabel("Tensão Aplicada (V)", fontsize=12)

    if not np.isnan(v0_global_mean):
        trans = axes[0].get_xaxis_transform()
        axes[0].annotate(
            f"{v0_global_mean:.2f}",
            xy=(v0_global_mean, 1.02),
            xycoords=trans,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            color="black",
        )

    plt.show()

In [ ]:
# Exemplo com Ultra-Violeta
target_prefix = "UV"
target_name = "Ultra-Violeta"
target_info = next(item for item in colors_info if item["name"] == target_name)

collected_curves = []

for intensity in [100, 80, 60, 40, 20]:
    fpath = os.path.join(data_dir, f"{target_info['prefix']}-{intensity}.csv")
    if os.path.exists(fpath):
        df_ex = pd.read_csv(fpath, sep=";", decimal=",")
        corrente_ex = df_ex["Corrente [A]"].values
        
        if df_sem is not None and len(corrente_ex) == len(df_sem):
            corrente_ex = corrente_ex - df_sem["Corrente [A]"].values
            
        tensao_ex = df_ex["Tensao [V]"].values
        collected_curves.append((tensao_ex, corrente_ex, f"{intensity}%"))

collected_curves.sort(key=lambda x: int(x[2].strip("%")), reverse=True)

if collected_curves:
    plot_method_4_stacked_origin_style(
        collected_curves,
        target_name,
        noise_region=(-9, -3),
        sigma_multiplier=2.0,
    )
else:
    print("Nenhuma curva encontrada para gerar o gráfico.")

### 6.2 Exemplo de Visualização (Ultra-Violeta)

## 6. Visualização Detalhada do Método 4

### 6.1 Função de Plotagem Empilhada

In [ ]:
def calculate_v0_method4_single(v, i, noise_region=(-9, -3), sigma_mult=2.0):
    """
    Calcula V₀ para uma única curva usando o método da 2ª derivada (Onset).
    
    Args:
        v: Array de tensões
        i: Array de correntes
        noise_region: Região para caracterização do ruído
        sigma_mult: Multiplicador do sigma para threshold
        
    Returns:
        float: Valor de V₀ ou np.nan se não encontrado
    """
    i_s = i * SCALE

    window = max(15, min(21, len(i) // 5))
    if window % 2 == 0:
        window += 1

    try:
        i_smooth = savgol_filter(i_s, window, 3)
        di = np.gradient(i_smooth, v)
        d2i = np.gradient(di, v)

        mask_noise = (v >= noise_region[0]) & (v <= noise_region[1])
        if np.sum(mask_noise) < 10:
            return np.nan

        res = stats.linregress(v[mask_noise], d2i[mask_noise])
        pred_noise_region = res.intercept + res.slope * v[mask_noise]
        std_noise = np.std(d2i[mask_noise] - pred_noise_region)

        mask_search = v > noise_region[1]
        v_search = v[mask_search]
        d2i_search = d2i[mask_search]

        pred_full = res.intercept + res.slope * v_search
        diff = np.abs(d2i_search - pred_full)
        threshold = sigma_mult * std_noise

        out_indices = np.where(diff > threshold)[0]

        if len(out_indices) > 0:
            return v_search[out_indices[0]]

    except Exception:
        return np.nan
        
    return np.nan

In [ ]:
summary_data = []

print(
    f"{'Cor':<15} | {'V0 Médio':<10} | {'Desvio Pad.':<12} | {'Erro Padrão':<12} | {'Inc. Total':<12} | {'N':<3}"
)
print("-" * 85)

for info in colors_info:
    v0_collected = []

    for intensity in intensities:
        fpath = os.path.join(data_dir, f"{info['prefix']}-{intensity}.csv")
        if not os.path.exists(fpath):
            continue

        df_temp = pd.read_csv(fpath, sep=";", decimal=",")

        if "df_sem" in globals() and len(df_temp) == len(df_sem):
            curr_vals = df_temp["Corrente [A]"].values - df_sem["Corrente [A]"].values
        else:
            curr_vals = df_temp["Corrente [A]"].values

        volt_vals = df_temp["Tensao [V]"].values

        v0 = calculate_v0_method4_single(
            volt_vals, curr_vals, noise_region=(-9, -3), sigma_mult=2.5
        )

        if not np.isnan(v0):
            v0_collected.append(v0)

    # Estatística
    if len(v0_collected) > 0:
        v0_arr = np.array(v0_collected)
        mean_v0 = np.mean(v0_arr)

        if len(v0_collected) > 1:
            std_v0 = np.std(v0_arr, ddof=1)
        else:
            std_v0 = 0.0

        sigma_m = std_v0 / np.sqrt(len(v0_collected))
        sigma_t = np.sqrt(sigma_m**2 + SIGMA_INST**2)

        summary_data.append(
            {
                "Cor": info["name"],
                "V0 (V)": mean_v0,
                "Sigma_V0": std_v0,
                "Sigma_m": sigma_m,
                "Sigma_Total": sigma_t,
                "N_Amostras": len(v0_collected),
            }
        )

        print(
            f"{info['name']:<15} | {mean_v0:7.4f} V | {std_v0:8.4f} V   | {sigma_m:8.4f} V   | {sigma_t:8.4f} V   | {len(v0_collected)}"
        )
    else:
        print(f"{info['name']:<15} | {'N/A':<10} | {'-'*12} | {'-'*12} | {'-'*12} | 0")

df_analytics = pd.DataFrame(summary_data)
df_analytics.set_index("Cor", inplace=True)

display(df_analytics)

### 7.2 Cálculo Estatístico para Todas as Cores

## 7. Análise Estatística Completa

### 7.1 Função para Cálculo de V₀ Individual

In [ ]:
def plot_planck_determination(df_results):
    """
    Plota V₀ x Frequência, faz regressão linear ponderada e determina h.
    Estilo 'Origin' conforme solicitado.
    
    Args:
        df_results: DataFrame com resultados estatísticos de V₀
        
    Returns:
        tuple: (h_medido, phi_medido) em eV·s e eV
    """

    # Preparação dos dados
    x_freq = []
    y_v0 = []
    y_err = []

    colors_ordered = [
        "Vermelho",
        "Amarelo",
        "Verde",
        "Azul",
        "Violeta",
        "Ultra-Violeta",
    ]

    for color in colors_ordered:
        if color in df_results.index:
            try:
                freq_val = frequencias[color]
                v0_val = df_results.loc[color, "V0 (V)"]
                err_val = df_results.loc[color, "Sigma_Total"]

                if not np.isnan(v0_val):
                    x_freq.append(freq_val)
                    y_v0.append(v0_val)
                    y_err.append(err_val)
            except KeyError:
                continue

    x = np.array(x_freq)
    y = np.array(y_v0)
    sigma = np.array(y_err)

    # Regressão linear ponderada
    def linear_model(x, a, b):
        return a * x + b

    popt, pcov = curve_fit(linear_model, x, y, sigma=sigma, absolute_sigma=True)

    slope = popt[0]
    intercept = popt[1]
    perr = np.sqrt(np.diag(pcov))
    slope_err = perr[0]
    intercept_err = perr[1]

    # Estatísticas de qualidade
    residuals = y - linear_model(x, *popt)
    ss_res = np.sum((residuals / sigma) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (np.sum(residuals**2) / ss_tot)
    pearson_r, _ = stats.pearsonr(x, y)

    # Constantes físicas
    h_measured_eVs = slope * 1e-14
    h_err_eVs = slope_err * 1e-14
    phi_measured_eV = -intercept
    phi_err_eV = intercept_err

    # Plotagem
    fig, ax = plt.subplots(figsize=(10, 8))

    ax.errorbar(
        x,
        y,
        yerr=sigma,
        fmt="o",
        color="black",
        ecolor="black",
        capsize=5,
        elinewidth=1,
        markersize=8,
        label="$V_0$ Medido",
    )

    x_fit = np.linspace(min(x) - 0.5, max(x) + 0.5, 100)
    y_fit = linear_model(x_fit, slope, intercept)
    ax.plot(x_fit, y_fit, color="#D32F2F", linewidth=2.5, label="Ajuste Linear")

    def get_confidence_interval(x_vals, popt, pcov, n_std=1.0):
        var_a = pcov[0, 0]
        var_b = pcov[1, 1]
        cov_ab = pcov[0, 1]
        sigma_y = np.sqrt(var_a * x_vals**2 + var_b + 2 * x_vals * cov_ab)
        return n_std * sigma_y

    y_conf = get_confidence_interval(x_fit, popt, pcov, n_std=1)
    ax.fill_between(
        x_fit,
        y_fit - y_conf,
        y_fit + y_conf,
        color="#D32F2F",
        alpha=0.2,
        label="Banda de Confiança 68%",
    )

    ax.set_xlabel(
        r"Frequência da Radiação ($\times 10^{14}$ Hz)", fontsize=14, fontstyle="italic"
    )
    ax.set_ylabel(r"Potencial de Corte $V_0$ (V)", fontsize=14, fontstyle="italic")

    # Tabela de estatísticas
    table_text = [
        ["Equation", "y = a + b*x"],
        ["Intercept", f"{intercept:.4f} ± {intercept_err:.4f}"],
        ["Slope", f"{slope:.4e} ± {slope_err:.4e}"],
        ["Pearson's r", f"{pearson_r:.5f}"],
        ["R-Square", f"{r_squared:.5f}"],
        ["Chi-Square", f"{ss_res:.4f}"],
    ]

    the_table = plt.table(
        cellText=table_text,
        colWidths=[0.3, 0.45],
        loc="lower right",
        bbox=[0.55, 0.15, 0.4, 0.25],
    )
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(10)
    the_table.scale(1, 1.3)

    ax.legend(
        loc="upper left", frameon=True, fontsize=11, fancybox=False, edgecolor="black"
    )

    ax.tick_params(direction="in", top=True, right=True, which="both")
    plt.grid(False)

    ax.set_xlim(min(x) - 0.5, max(x) + 0.5)
    ax.set_ylim(min(y) - 0.2, max(y) + 0.2)

    plt.tight_layout()
    plt.show()

    # Resultados numéricos
    print("\n" + "=" * 50)
    print("DETERMINAÇÃO DA CONSTANTE DE PLANCK")
    print("=" * 50)
    print(f"Inclinação da Reta (Slope): {slope:.5f} ± {slope_err:.5f} [V / 10^14 Hz]")
    print(f"Intercepto Linear: {intercept:.5f} ± {intercept_err:.5f} [V]")
    print("-" * 50)
    print(f"Constante de Planck (Medida):")
    print(f"h = {h_measured_eVs:.4e} ± {h_err_eVs:.4e} eV·s")
    print("-" * 50)
    print(f"Função Trabalho (Média do Material):")
    print(f"Φ = {phi_measured_eV:.4f} ± {phi_err_eV:.4f} eV")
    print("=" * 50)

    return h_measured_eVs, phi_measured_eV

In [ ]:
if "df_analytics" in globals():
    h_result, phi_result = plot_planck_determination(df_analytics)
else:
    print("Por favor, execute a célula anterior para gerar 'df_analytics' primeiro.")

### 8.2 Execução da Análise

## 8. Determinação da Constante de Planck

### 8.1 Função de Análise e Plotagem